In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

In [ ]:
df4 = pd.read_pickle('../data/df4.pkl')

In [ ]:
#df4.head(5)

In [ ]:
#df4.columns

In [ ]:
df4.shape

In [ ]:
df4_with_regions = pd.read_pickle('../data/df4_with_regions.pkl')

In [ ]:
df5 = pd.read_pickle('../data/df5.pkl')

## For Visualisation Site

Values of Current Transfer Timing Distribution by Age Group and Time of Day
- Student (Primary 1 to JC/Poly) --> 7 - 19 years old
- Adult --> 20 - 59 years old
- Senior Citizen --> 60 years and above

In [ ]:
def trf_time_distribution_csv(data):
    df = data[data['final_termination_reason'] == 'candidate_transfer'].copy()

    df['age_group'] = df['PATRON_CATG_DESC_TXT'].map({
        'Student': '7 - 19',
        'Adult': '20-59',
        'Senior Citizen': '60 and above'
    })

    df['next_entry_tm_hr'] = df['next_ENTRY_TM'].dt.hour

    result = df.pivot_table(
        index = 'age_group',
        columns = 'next_entry_tm_hr',
        aggfunc = 'size',
        fill_value = 0
    )

    result.to_csv('../data/trf_time_distribution.csv', index=False)

In [ ]:
# to test
trf_time_distribution_csv(df4) # are we going to change the dataset? since this is only a certain range

In [ ]:
def trf_time_distribution(file):
    df = pd.read_csv(file)
    return df

In [ ]:
trf_time_distribution('../data/trf_time_distribution.csv')

Distribution of Spatial Transfer Location Pairs

In [ ]:
def trf_pair_distribution_csv(data):
    df = data[data['final_termination_reason'] == 'candidate_transfer']

    result = df.pivot_table(
        index = 'ORIG_STATION_NAME',
        columns = 'DEST_STATION_NAME',
        aggfunc = 'size',
        fill_value = 0
    )

    # use this if want proportions instead
    # result = (
    # df.groupby(['ORIG_STATION_NAME', 'DEST_STATION_NAME'])
    #   .size()
    #   .groupby(level = 0)
    #   .apply(lambda x: x / x.sum())
    #   .reset_index(name = "value")
    # )

    result.to_csv('../data/trf_pair_dist.csv', index=False)

In [ ]:
# to test
trf_pair_distribution_csv(df4) # are we going to change the dataset? since this is only a certain range

In [ ]:
def trf_pair_distribution(file):
    df = pd.read_csv(file)
    return df

In [ ]:
trf_pair_distribution('../data/trf_pair_dist.csv')

Temporal Transfer Pattern Distribution (y-axis: Average Transfer Time, x-axis: Hour of Day)

In [ ]:
def trf_pattern_distribution(data):
    df = data[data['final_termination_reason'] == 'candidate_transfer']

    df['next_entry_tm_hr'] = df['next_ENTRY_TM'].dt.hour

    trf_avg = (
        df.groupby('next_entry_tm_hr')['time_gap_mins']
        .mean()
        .reset_index(name = 'average_transfer_time')
    )

    plt.figure(figsize=(8, 5))

    plt.bar(trf_avg['next_entry_tm_hr'], trf_avg['average_transfer_time'])

    plt.title("Average Transfer Time")
    plt.xlabel("Hour of Day")
    plt.ylabel("Average Transfer Time by Hour of Day")

    plt.xticks(range(0, 24))

    plt.show()

In [ ]:
# to test
trf_pattern_distribution(df4) # are we going to change the dataset? since this is only a certain range

## For Geospatial Analysis (Delay Simulation Tab)

Identification of Misclassified Transfers According to our Model - Regional Hotspots (Most Affected Regions)

In [ ]:
def misclassified_regional_csv(data, correct_data):
    df = data[['CRD_NUM', 'JRNY_ID_NUM', 'ENTRY_TM', 'EXIT_TM', 'TRNSPT_MODE_CD', 'journey_termination_flag', 'ORIG_STATION_NAME', 'dest_region']]
    correct_df = correct_data[['CRD_NUM', 'JRNY_ID_NUM', 'JRNY_START_TM', 'JRNY_END_TM', 'PATRON_CATG_ID_NUM']]

    correct_df['service_day'] = (correct_df['JRNY_START_TM'] - pd.Timedelta(hours=5)).dt.date
    target_day = pd.Timestamp('2025-02-12').date()
    correct_df = correct_df[correct_df['service_day'] == target_day].reset_index(drop=True)
    correct_df = correct_df[correct_df['PATRON_CATG_ID_NUM'].isin([1, 3, 4])].reset_index(drop=True)

    df_val = df.merge(
        correct_df[['CRD_NUM', 'JRNY_ID_NUM', 'JRNY_START_TM', 'JRNY_END_TM']],
        on=['CRD_NUM', 'JRNY_ID_NUM'],
        how='inner'
    )

    df_val = df_val[
        (df_val['ENTRY_TM'] >= df_val['JRNY_START_TM']) &
        (df_val['EXIT_TM'] <= df_val['JRNY_END_TM'])
    ].reset_index(drop=True)

    df_val = df_val.sort_values(['CRD_NUM', 'ENTRY_TM']).reset_index(drop=True)
    df_val['next_JRNY_ID_NUM'] = df_val.groupby('CRD_NUM')['JRNY_ID_NUM'].shift(-1)
    df_val['true_transfer'] = (df_val['JRNY_ID_NUM'] == df_val['next_JRNY_ID_NUM'])
    df_val = df_val[df_val['next_JRNY_ID_NUM'].notna()].reset_index(drop=True)

    actual_transfer = df_val['true_transfer']
    pred_transfer = ~df_val['journey_termination_flag']

    fp = (~actual_transfer & pred_transfer) # actual new journey, predicted transfer (merge error)
    fn = (actual_transfer & ~pred_transfer) # actual transfer, predicted new journey (split error)

    fp_df = df_val[fp]
    fn_df = df_val[fn]

    fp_by_region = fp_df.groupby('dest_region')
    fn_by_region = fn_df.groupby('dest_region')

    fp_by_region.to_csv('../data/fp_by_region.csv', index=False)
    fn_by_region.to_csv('../data/fn_by_region.csv', index=False)

In [ ]:
misclassified_regional_csv(df4_with_regions, df5)

In [ ]:
def misclassified_regional(fp_csv, fn_csv):
    fp = pd.read_csv(fp_csv)
    fn = pd.read_csv(fn_csv)
    return fp, fn

In [ ]:
misclassified_regional('../data/fp_by_region.csv', '../data/fn_by_region.csv')

Identification of Misclassified Transfers According to our Model - Exact Bus Stops/Transfer Pairs

In [ ]:
def misclassified_pair_csv(data, correct_data):
    df = data[['CRD_NUM', 'JRNY_ID_NUM', 'ENTRY_TM', 'EXIT_TM', 'TRNSPT_MODE_CD', 'journey_termination_flag', 'ORIG_STATION_NAME', 'next_orig_station']]
    correct_df = correct_data[['CRD_NUM', 'JRNY_ID_NUM', 'JRNY_START_TM', 'JRNY_END_TM', 'PATRON_CATG_ID_NUM']]

    correct_df['service_day'] = (correct_df['JRNY_START_TM'] - pd.Timedelta(hours=5)).dt.date
    target_day = pd.Timestamp('2025-02-12').date()
    correct_df = correct_df[correct_df['service_day'] == target_day].reset_index(drop=True)
    correct_df = correct_df[correct_df['PATRON_CATG_ID_NUM'].isin([1, 3, 4])].reset_index(drop=True)

    df_val = df.merge(
        correct_df[['CRD_NUM', 'JRNY_ID_NUM', 'JRNY_START_TM', 'JRNY_END_TM']],
        on=['CRD_NUM', 'JRNY_ID_NUM'],
        how='inner'
    )

    df_val = df_val[
        (df_val['ENTRY_TM'] >= df_val['JRNY_START_TM']) &
        (df_val['EXIT_TM'] <= df_val['JRNY_END_TM'])
    ].reset_index(drop=True)

    df_val = df_val.sort_values(['CRD_NUM', 'ENTRY_TM']).reset_index(drop=True)
    df_val['next_TRNSPT_MODE_CD'] = df_val.groupby('CRD_NUM')['TRNSPT_MODE_CD'].shift(-1)
    df_val['next_JRNY_ID_NUM'] = df_val.groupby('CRD_NUM')['JRNY_ID_NUM'].shift(-1)
    df_val['true_transfer'] = (df_val['JRNY_ID_NUM'] == df_val['next_JRNY_ID_NUM'])
    df_val = df_val[df_val['next_JRNY_ID_NUM'].notna()].reset_index(drop=True)

    df_val["mode_pair"] = np.select(
        [
            (df_val['TRNSPT_MODE_CD'] == 1) & (df_val['next_TRNSPT_MODE_CD'] == 1),
            (df_val['TRNSPT_MODE_CD'] == 2) & (df_val['next_TRNSPT_MODE_CD'] == 1),
            (df_val['TRNSPT_MODE_CD'] == 1) & (df_val['next_TRNSPT_MODE_CD'] == 2),
            (df_val['TRNSPT_MODE_CD'] == 2) & (df_val['next_TRNSPT_MODE_CD'] == 2)
        ],
        [
            'bus_bus',
            'train_bus',
            'bus_train',
            'train_train'
        ],
        default="other"
    )    

    actual_transfer = df_val['true_transfer']
    pred_transfer = ~df_val['journey_termination_flag']

    fp = (~actual_transfer & pred_transfer) # actual new journey, predicted transfer (merge error)
    fn = (actual_transfer & ~pred_transfer) # actual transfer, predicted new journey (split error)

    fp_df = df_val[fp]
    fn_df = df_val[fn]

    fp_pair = fp_df.groupby(['orig_region', 'ORIG_STATION_NAME', 'next_orig_station']).size().reset_index(name='count').sort_values(by='count', ascending=False)
    fn_pair = fn_df.groupby(['orig_region', 'ORIG_STATION_NAME', 'next_orig_station']).size().reset_index(name='count').sort_values(by='count', ascending=False)

    fp_pair.to_csv('../data/fp_pair.csv', index=False)
    fn_pair.to_csv('../data/fn_pair.csv', index=False)    

In [ ]:
misclassified_pair_csv(df4, df5)

In [ ]:
def misclassified_pair(fp_csv, fn_csv):
    fp = pd.read_csv(fp_csv)
    fn = pd.read_csv(fn_csv)
    return fp, fn

In [ ]:
misclassified_pair('../data/fp_pair.csv', '../data/fn_pair.csv')

## For Policy Simulator

Input Age/Demographic, Time of Day, Tap-in and Tap-out regions, Output Recommended Transfer Time Change, Benefits and Costs

In [ ]:
# _base_dir = os.path.dirname(os.path.abspath(__file__)) # This line is for py file
# _data_dir = os.path.join(_base_dir, '..', 'data')
_data_dir = '../data'
 
def get_welfare_summary(patron, window, spec):
    """
    Look up welfare results for a given patron category, transfer window, and classifier spec.
 
    Inputs:
    - patron: 'Overall', 'Adult', 'Student', 'Senior Citizen'
    - window: integer, one of 20, 25, 30, 35, 40, 45, 50, 55
    - spec: 'strict', 'baseline', 'lenient'
 
    Returns:
    - dict with split/merge error counts, unconditional rates, and conditional rates
    """
    final_results_df = pd.read_csv(os.path.join(_data_dir, 'welfare_results.csv'))
 
    row = final_results_df[
        (final_results_df['patron'] == patron) &
        (final_results_df['window_mins'] == window) &
        (final_results_df['spec'] == spec)
    ]
 
    if row.empty:
        return f"No data found for patron='{patron}', window={window}, spec='{spec}'"
 
    row = row.iloc[0]
 
    return {
        'spec':                     spec,
        'patron':                   patron,
        'window_mins':              window,
        'classifier_transfer_n':    row['classifier_transfer_n'],
        'classifier_new_journey_n': row['classifier_new_journey_n'],
        'window_transfer_n':        row['window_transfer_n'],
        'window_new_journey_n':     row['window_new_journey_n'],
        # split error: window too strict, breaks classifier-valid transfers → commuter overcharged
        'split_error_n':            row['wrongly_split_n'],
        'split_error_pct':          row['wrongly_split_pct'],          # % of all pairs
        'split_error_cond_pct':     row['wrongly_split_cond_pct'],     # % of classifier-said transfers
        # merge error: window too lenient, links classifier-separate journeys → fare undercharged
        'merge_error_n':            row['wrongly_merged_n'],
        'merge_error_pct':          row['wrongly_merged_pct'],         # % of all pairs
        'merge_error_cond_pct':     row['wrongly_merged_cond_pct'],    # % of classifier-said new journeys
    }
 
 
def get_marginal_summary(patron, spec, window_from):
    """
    Look up the marginal welfare effect of increasing the transfer window by 5 minutes.
 
    Inputs:
    - patron: 'Overall', 'Adult', 'Student', 'Senior Citizen'
    - spec: 'strict', 'baseline', 'lenient'
    - window_from: integer, one of 20, 25, 30, 35, 40, 45, 50
                   (returns the effect of moving from window_from to window_from + 5)
 
    Returns:
    - dict with newly linked pairs, marginal benefit (legitimate transfers rescued),
      and marginal cost (illegitimate links added)
    """
    marginal_df = pd.read_csv(os.path.join(_data_dir, 'welfare_marginal.csv'))
 
    window_to = window_from + 5
 
    row = marginal_df[
        (marginal_df['patron'] == patron) &
        (marginal_df['spec'] == spec) &
        (marginal_df['window_from'] == window_from) &
        (marginal_df['window_to'] == window_to)
    ]
 
    if row.empty:
        return f"No data found for patron='{patron}', spec='{spec}', window_from={window_from}"
 
    row = row.iloc[0]
 
    return {
        'spec':               spec,
        'patron':             patron,
        'window_from':        window_from,
        'window_to':          window_to,
        'newly_linked_n':     row['newly_linked_n'],       # total pairs newly linked by the larger window
        'marginal_benefit_n': row['marginal_benefit_n'],   # newly linked where classifier says transfer → legitimate rescue
        'marginal_cost_n':    row['marginal_cost_n'],      # newly linked where classifier says new journey → illegitimate link
    }
 

In [ ]:
# test get_welfare_summary
print(get_welfare_summary('Adult', 35, 'baseline'))

In [ ]:
# test get_marginal_summary
print(get_marginal_summary('Adult', 'baseline', 35))

In [ ]:
def query_delay_sim(
    delay_mins:      int,
    bus_window:      int,
    classifier_type: str,
    patron:          str = 'all',
    df:              pd.DataFrame = None
):
    if df is None:
        raise ValueError("Please provide final_df")

    valid_delays  = [0, 5, 10, 15, 20]
    valid_windows = list(range(35, 65, 5))
    valid_specs   = ['baseline', 'lenient', 'strict']

    if delay_mins      not in valid_delays:  raise ValueError(f"delay_mins must be one of {valid_delays}")
    if bus_window      not in valid_windows: raise ValueError(f"bus_window must be one of {valid_windows}")
    if classifier_type not in valid_specs:   raise ValueError(f"classifier_type must be one of {valid_specs}")

    sub = df[
        (df['delay_mins']      == delay_mins)  &
        (df['bus_window_mins'] == bus_window)  &
        (df['spec']            == classifier_type)
    ].copy()

    if sub.empty:
        raise ValueError("No data found for given parameters")

    if patron == 'all':
        main_row = sub[sub['breakdown_type'] == 'overall'].iloc[0]
    else:
        patron_rows = sub[sub['breakdown_type'] == 'patron']
        if patron not in patron_rows['breakdown_value'].values:
            raise ValueError(f"Patron '{patron}' not found. Available: {patron_rows['breakdown_value'].tolist()}")
        main_row = patron_rows[patron_rows['breakdown_value'] == patron].iloc[0]

    classifier_journeys = int(main_row['classifier_journeys']) if not pd.isna(main_row['classifier_journeys']) else None
    window_journeys     = int(main_row['window_journeys'])     if not pd.isna(main_row['window_journeys'])     else None
    journey_difference  = (window_journeys - classifier_journeys) if (window_journeys and classifier_journeys) else None

    def get_breakdown(breakdown_type, col_name):
        return (
            sub[sub['breakdown_type'] == breakdown_type][[
                'breakdown_value', 'n_pairs', 'n_cards',
                'classifier_journeys', 'window_journeys',
                'wrongly_split_n',       'wrongly_merged_n',
                'wrongly_split_pct',     'wrongly_merged_pct',
                'wrongly_split_pct_all', 'wrongly_merged_pct_all',
            ]]
            .rename(columns={'breakdown_value': col_name})
            .sort_values('wrongly_split_pct', ascending=False)
            .reset_index(drop=True)
        )

    return {
        'spec':                classifier_type,
        'delay_mins':          delay_mins,
        'bus_window_mins':     bus_window,
        'patron':              patron,
        'classifier_journeys': classifier_journeys,
        'window_journeys':     window_journeys,
        'journey_difference':  journey_difference,
        'wrongly_split_n':     int(main_row['wrongly_split_n']),
        'wrongly_merged_n':    int(main_row['wrongly_merged_n']),
        'wrongly_split_pct':   float(main_row['wrongly_split_pct']),
        'wrongly_merged_pct':  float(main_row['wrongly_merged_pct']),
        'by_patron':           get_breakdown('patron',          'patron'),
        'by_dest_region':      get_breakdown('dest_region',     'dest_region'),
        'by_orig_region':      get_breakdown('orig_region',     'orig_region'),
        'by_hour':             get_breakdown('next_entry_hour', 'hour'),
    }



In [ ]:
final_df = pd.read_csv('../data/delay_sim_results.csv')

In [ ]:
# call the function
result = query_delay_sim(
    delay_mins      = 10,       # 0, 5, 10, 15, 20
    bus_window      = 45,       # 35, 40, 45, 50, 55, 60
    classifier_type = 'baseline',  # 'baseline', 'lenient', 'strict'
    patron          = 'all',    # 'all', 'Adult', 'Student', 'Senior Citizen'
    df              = final_df
)